<a href="https://colab.research.google.com/github/MrTropicalMango/Maestria_IA_Proyecto_Titulacion/blob/main/Demo_MVP_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UNIR

**Seminario de Innovación en Inteligencia Artificial**

**Equipo 3013B**

**Integrantes:** Durán García Sergio Israel; Mallorquín Cabral Christian Javier; Mejía Sandoval Octavio Francisco; Velásquez Irías Fausto Rafael.

---

# Demo del MVP de clasificación presuntiva

Este notebook permite a cualquier integrante del equipo cargar una imagen pública o desidentificada, obtener una clasificación presuntiva, visualizar Grad-CAM y registrar una revisión humana.

**Alcance:** demostración académica. No es diagnóstico, no sustituye al personal QBP y no debe usarse con datos de pacientes. El notebook usa Google Drive para conservar el modelo y la bitácora CSV.Drive para conservar el modelo y la bitácora CSV.

In [ ]:
# Dependencias: se usa PyTorch preinstalado en Colab; el demo no usa torchvision.
%pip install -q gradio==5.44.1
from google.colab import drive
drive.mount('/content/drive')

# Recuperar los pesos del paquete de evidencia compartido, sin sobrescribirlos.
from pathlib import Path
from zipfile import ZipFile
weights_path = Path('/content/drive/MyDrive/Colab Notebooks/Seminario_Titulacion_MVP_CNN/resultados_cnn_cv5/modelo_seleccionado.pt')
if not weights_path.is_file():
    package_path = Path('/content/drive/MyDrive/Paquete_Evidencia_Tecnica_sin_imagenes.zip')
    with ZipFile(package_path) as package:
        matches = [n for n in package.namelist() if n.endswith('resultados_cnn_cv5/modelo_seleccionado.pt')]
        if len(matches) != 1:
            raise RuntimeError(f'Se esperaba un modelo único: {matches}')
        weights_path.parent.mkdir(parents=True, exist_ok=True)
        with weights_path.open('xb') as target:
            target.write(package.read(matches[0]))
    print('Pesos recuperados del paquete de evidencia:', weights_path)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 757.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.6/324.6 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have hu

In [ ]:
import base64, csv, io, json, os, glob
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import gradio as gr

torch.set_num_threads(1)
LABELS = ['Acinetobacter baumannii', 'Actinomyces israelii', 'Bacteroides fragilis', 'Enterococcus faecalis', 'Escherichia coli']
DRIVE_ROOT = Path('/content/drive/MyDrive')
WEIGHTS = DRIVE_ROOT / 'Colab Notebooks/Seminario_Titulacion_MVP_CNN/resultados_cnn_cv5/modelo_seleccionado.pt'
if not WEIGHTS.is_file():
    raise FileNotFoundError(f'No se encontró el modelo en: {WEIGHTS}. Verifica la carpeta compartida del proyecto.')
BITACORA = WEIGHTS.parent / 'bitacora_demo_colab.csv'
print('Modelo:', WEIGHTS)
print('Bitácora:', BITACORA)


Modelo: /content/drive/MyDrive/Colab Notebooks/Seminario_Titulacion_MVP_CNN/resultados_cnn_cv5/modelo_seleccionado.pt
Bitácora: /content/drive/MyDrive/Colab Notebooks/Seminario_Titulacion_MVP_CNN/resultados_cnn_cv5/bitacora_demo_colab.csv


In [ ]:
class StandardCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.block2 = nn.Sequential(nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.last_conv = nn.Conv2d(64, 96, 3, padding=1)
        self.head = nn.Sequential(nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.20), nn.Linear(96, 5))
    def forward(self, x): return self.head(self.last_conv(self.block2(self.block1(x))))

checkpoint = torch.load(WEIGHTS, map_location='cpu', weights_only=False)
model = StandardCNN()
model.load_state_dict(checkpoint['state_dict'])
model.eval()
print('Arquitectura:', checkpoint.get('architecture', 'cnn_estandar'))
print('Modelo cargado correctamente')

Arquitectura: cnn_estandar
Modelo cargado correctamente


In [ ]:
def infer(image):
    if image is None: return 'Carga una imagen PNG o JPEG.', None, None
    image = Image.fromarray(image).convert('RGB') if isinstance(image, np.ndarray) else image.convert('RGB')
    if image.width < 224 or image.height < 224: return 'La imagen debe medir al menos 224 × 224 píxeles.', None, None
    display = image.copy()
    arr = np.asarray(image.resize((96, 96)), dtype=np.float32).transpose(2, 0, 1) / 255.0
    tensor = torch.tensor(arr).unsqueeze(0)
    activations, gradients = [], []
    ha = model.last_conv.register_forward_hook(lambda m, i, o: activations.append(o))
    hg = model.last_conv.register_full_backward_hook(lambda m, gi, go: gradients.append(go[0]))
    logits = model(tensor)
    probabilities = torch.softmax(logits, dim=1)[0]
    predicted = int(probabilities.argmax().item())
    model.zero_grad(); logits[0, predicted].backward()
    weights = gradients[0].mean((2, 3), keepdim=True)
    cam = (weights * activations[0]).sum(1).relu()[0].detach().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    mask = Image.fromarray(np.uint8(cam * 255)).resize(display.size).point(lambda v: v * .70)
    overlay = Image.blend(display, Image.new('RGB', display.size, (255, 0, 0)), .42)
    heatmap = Image.composite(overlay, display, mask)
    ha.remove(); hg.remove()
    confidence = float(probabilities[predicted])
    text = f'Predicción presuntiva: {LABELS[predicted]}\nConfianza relativa: {confidence:.1%}\nRevisión QBP requerida.'
    return text, heatmap, {'archivo': 'imagen_colab', 'modelo': checkpoint.get('architecture', 'cnn_estandar'), 'prediccion': LABELS[predicted], 'confianza': round(confidence, 6)}

In [ ]:
def save_review(result, action, correction, note):
    if not result: return 'Primero analiza una imagen.'
    final = correction if action == 'corregir' else (result['prediccion'] if action == 'confirmar' else '')
    if action == 'corregir' and not correction: return 'Selecciona la clase corregida.'
    row = {**result, 'fecha_utc': datetime.now(timezone.utc).isoformat(), 'accion': action, 'resultado_final': final, 'nota': note or ''}
    exists = BITACORA.exists()
    with BITACORA.open('a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if not exists: writer.writeheader()
        writer.writerow(row)
    return f'Revisión guardada en Drive: {BITACORA.name}'

In [ ]:
with gr.Blocks(title='Demo MVP microbiología') as demo:
    gr.Markdown('## Demo académico — clasificación presuntiva\nNo usar con datos de pacientes. Toda predicción requiere revisión profesional.')
    with gr.Row():
        image = gr.Image(type='numpy', label='Imagen pública o desidentificada')
        with gr.Column():
            result = gr.Textbox(label='Resultado')
            heatmap = gr.Image(label='Grad-CAM')
    analyze = gr.Button('Analizar imagen')
    state = gr.State()
    analyze.click(infer, image, [result, heatmap, state])
    gr.Markdown('### Revisión humana')
    action = gr.Radio(['confirmar', 'corregir', 'no_concluyente'], value='confirmar', label='Acción')
    correction = gr.Dropdown(LABELS, label='Clase corregida', allow_custom_value=False)
    note = gr.Textbox(label='Nota opcional, sin datos personales')
    save = gr.Button('Guardar revisión en Drive')
    status = gr.Textbox(label='Estado')
    save.click(save_review, [state, action, correction, note], status)
demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://84c99f348d9df8acfb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
